In [1]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Kelly Quinn kellyquinn@berkeley.edu"}  # SEC requires a real contact
CIK = "0000771497"  # ABM Industries

def filing_index():
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{CIK}.json", headers=HEADERS).json()
    frames = [pd.DataFrame(sub["filings"]["recent"])]
    for f in sub["filings"].get("files", []):  # older filings live in paged files
        frames.append(pd.DataFrame(
            requests.get(f"https://data.sec.gov/submissions/{f['name']}", headers=HEADERS).json()))
        time.sleep(0.2)
    df = pd.concat(frames, ignore_index=True).query("form == '10-K'").copy()
    acc = df["accessionNumber"].str.replace("-", "")
    base = "https://www.sec.gov/Archives/edgar/data/771497/" + acc + "/"
    # some old filings have no primary document; fall back to the full .txt submission
    df["url"] = base + df["primaryDocument"].where(df["primaryDocument"] != "",
                                                    df["accessionNumber"] + ".txt")
    return df[["reportDate", "filingDate", "url"]].sort_values("reportDate").reset_index(drop=True)

def get_text(url):
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return re.sub(r"\s+", " ", BeautifulSoup(r.text, "lxml").get_text(" "))

NUM = r"([\d,]{4,})"
PATTERNS = {
    "employees": rf"employ(?:s|ed)?\s+(?:approximately|over|more than|about)?\s*{NUM}\s+(?:persons|employees|people)",
    "cba_count": rf"{NUM}\s*(?:of these employees|employees)?[^.]{{0,60}}?(?:covered|subject)[^.]{{0,40}}collective bargaining",
    "cba_pct":   r"(\d{1,2})\s*%[^.]{0,40}(?:were |are )?(?:covered|subject)[^.]{0,40}collective bargaining",
    "mgmt":      rf"(?:about|approximately)\s+{NUM}\s+employees with executive",
    "jan_share": r"Janitorial accounted for (?:nearly |approximately |about )?(\d{1,2})\s*%",
}

def snippet(text, key, before=300, after=150):
    i = text.lower().find(key)
    return text[max(0, i - before): i + after] if i >= 0 else None

rows = []
for f in filing_index().itertuples():
    text = get_text(f.url)
    time.sleep(0.2)  # SEC asks for no more than 10 requests per second
    row = {"report_date": f.reportDate, "filing_date": f.filingDate, "url": f.url}
    for name, pat in PATTERNS.items():
        m = re.search(pat, text, flags=re.I)
        row[name] = m.group(1) if m else None
    row["cba_snippet"] = snippet(text, "collective bargaining")
    row["janitorial_snippet"] = snippet(text, "janitorial accounted for")
    rows.append(row)
    print(f.reportDate, row["employees"], row["cba_count"], row["cba_pct"])

abm = pd.DataFrame(rows)
for col in ["employees", "cba_count", "mgmt"]:
    abm[col] = pd.to_numeric(abm[col].str.replace(",", ""), errors="coerce")
abm["cba_share"] = (abm["cba_count"] / abm["employees"]).round(3)

abm.to_csv("../outputs/py/abm_10k_series.csv", index=False)

/var/folders/vg/ynj3b7_n2k55q4yx14xb5rmc0000gn/T/ipykernel_45927/1025362568.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True).query("form == '10-K'").copy()


1994-10-31 42,000 17,800 None
1998-10-31 55,000 25,700 None
2001-10-31 60,000 24,900 None
2002-10-31 62,000 27,800 None
2003-10-31 64,000 28,000 44
2004-10-31 70,000 29,000 41
2005-10-31 73,000 29,000 40
2006-10-31 75,000 29,000 39
2007-10-31 107,000 39,000 36
2008-10-31 100,000 43,000 44
2009-10-31 91,000 38,000 42
2010-10-31 96,000 40,000 43
2011-10-31 96,000 42,000 43
2012-10-31 95,000 46,000 48
2013-10-31 110,000 47,000 43
2014-10-31 118,000 59,000 50
2015-10-31 120,000 66,000 55
2016-10-31 110,000 2016, 40
2017-10-31 140,000 140,000 32
2018-10-31 140,000 140,000 34


/var/folders/vg/ynj3b7_n2k55q4yx14xb5rmc0000gn/T/ipykernel_45927/1025362568.py:28: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  return re.sub(r"\s+", " ", BeautifulSoup(r.text, "lxml").get_text(" "))


2019-10-31 140,000 140,000 32
2020-10-31 114,000 114,000 32
2021-10-31 124,000 124,000 35
2022-10-31 127,000 127,000 35
2023-10-31 123,000 123,000 37
2024-10-31 117,000 117,000 42
2025-10-31 113,000 113,000 45


In [2]:
abm = pd.read_csv("../outputs/py/abm_10k_series.csv")

# Union count from the "of which/whom approximately X, or Y%" sentence (2016 onward)
recent = abm["cba_snippet"].str.extract(
    r"of (?:which|whom)\s+approximately\s+([\d,]+)\s*,?\s*or\s+(\d+)\s*%")
abm["cba_count_fixed"] = pd.to_numeric(
    recent[0].str.replace(",", ""), errors="coerce").fillna(abm["cba_count"])
bad = abm["cba_count_fixed"] >= abm["employees"]
abm.loc[bad, "cba_count_fixed"] = pd.NA

abm["union_share"] = (abm["cba_pct"] / 100).fillna(abm["cba_count_fixed"] / abm["employees"])
abm_clean = abm.loc[abm["report_date"] >= "2001",
                    ["report_date", "employees", "cba_count_fixed", "union_share", "url"]]
abm_clean.to_csv("../outputs/py/abm_10k_clean.csv", index=False)
print(abm_clean.to_string(index=False))

report_date  employees  cba_count_fixed  union_share                                                                                   url
 2001-10-31      60000          24900.0     0.415000     https://www.sec.gov/Archives/edgar/data/771497/000095014901501984/f77849e10-k.txt
 2002-10-31      62000          27800.0     0.448387     https://www.sec.gov/Archives/edgar/data/771497/000095014902002436/f86502e10vk.htm
 2003-10-31      64000          28000.0     0.440000     https://www.sec.gov/Archives/edgar/data/771497/000095014904000137/f95384e10vk.htm
 2004-10-31      70000          29000.0     0.410000     https://www.sec.gov/Archives/edgar/data/771497/000095013405000777/f04468e10vk.htm
 2005-10-31      73000          29000.0     0.400000     https://www.sec.gov/Archives/edgar/data/771497/000095013406006158/f16287e10vk.htm
 2006-10-31      75000          29000.0     0.390000     https://www.sec.gov/Archives/edgar/data/771497/000095013406023656/f25881e10vk.htm
 2007-10-31     107000     

In [ ]:
abm_final = pd.read_csv("../outputs/py/abm_10k_clean.csv")
abm_final["count_derived"] = False
is2016 = abm_final["report_date"].str.startswith("2016")
abm_final.loc[is2016, "cba_count_fixed"] = round(0.40 * 110000)  # 44,000; filing reports % only
abm_final.loc[is2016, "count_derived"] = True
abm_final.to_csv("../outputs/py/abm_10k_clean.csv", index=False)
abm_final